In [0]:
%pip install aiohttp
%pip install aiolimiter

In [0]:
import os
import json
import time
import random
import aiohttp
import asyncio
import itertools
import nest_asyncio
from urllib.parse import quote
from collections import Counter
from aiolimiter import AsyncLimiter
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

In [0]:
prod_url = "https://api.tibiadata.com/v5/"

base_dir="/Volumes/tibia_analytics/bronze/raw/"
headers = {
        "User-Agent": "TibiaAnalytics (+https://github.com/vitorfehelberg/tibia-analytics/)"
    }

RATE_LIMIT = 32
MAX_CONCURRENCY = 96
limiter = AsyncLimiter(RATE_LIMIT, 1)
concurrency_limit = asyncio.Semaphore(MAX_CONCURRENCY)

http_stats = Counter()
latencies = []

nest_asyncio.apply()

In [0]:
RETRYABLE_STATUS = frozenset({429, 500, 502, 503, 504})
MAX_RETRY_DELAY = 30.0

def _parse_retry_after(value):
    if not value:
        return None
    try:
        return float(value)
    except ValueError:
        pass
    try:
        return (parsedate_to_datetime(value) - datetime.now(timezone.utc)).total_seconds()
    except (TypeError, ValueError):
        return None
    
def retry_delay(attempt, retry_after=None):
    seconds = _parse_retry_after(retry_after)
    if seconds is None:
        seconds = (2 ** attempt) + random.uniform(0.5, 1.5)
    return min(max(seconds, 0.0), MAX_RETRY_DELAY)

async def request(session, limiter, endpoint, retries=3):
    url = prod_url + endpoint
 
    for attempt in range(retries + 1):
        try:
            async with concurrency_limit, limiter:
                started = time.monotonic()
                async with session.get(url) as response:
                    latencies.append(time.monotonic() - started)
                    http_stats[response.status] += 1
                    if response.status < 400:
                        return await response.json()
                    if response.status not in RETRYABLE_STATUS or attempt == retries:
                        response.raise_for_status()
                    delay = retry_delay(attempt, response.headers.get("Retry-After"))
        except (aiohttp.ClientConnectionError, aiohttp.ClientPayloadError, asyncio.TimeoutError) as e:
            http_stats[type(e).__name__] += 1    
            if attempt == retries:
                raise
            delay = retry_delay(attempt)
 
        await asyncio.sleep(delay)

def print_http_summary(elapsed):
    ls = sorted(latencies)
    pct = lambda p: ls[min(int(len(ls) * p), len(ls) - 1)] if ls else 0.0
    attempts = sum(http_stats.values())
    print(f"{attempts} attempts in {elapsed:.0f}s = {attempts / elapsed:.1f} req/s | "
          f"latency p50={pct(.5):.2f}s p95={pct(.95):.2f}s p99={pct(.99):.2f}s")
    print("status/errors:", dict(http_stats.most_common()))


In [0]:
def save_api_to_json(data, folder, filename):
    path = os.path.join(base_dir, folder)
    os.makedirs(path, exist_ok=True)

    filepath = os.path.join(path, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

In [0]:
async def save_async(data, folder, filename):
    await asyncio.to_thread(save_api_to_json, data, folder, filename)

In [0]:
async def get_worlds(session, limiter):
    return await request(session, limiter, "worlds")

In [0]:
async def get_highscores(session, limiter, world, category="experience", vocation="all", page=1):
    endpoint = f"highscores/{world}/{category}/{vocation}/{page}"
    return await request(session, limiter, endpoint)

In [0]:
async def get_characters(session, limiter, name):
    try:
        return await request(session, limiter, f"character/{quote(name, safe='')}")
    except aiohttp.ClientResponseError as e:
        if e.status == 404:
            return None
        raise

In [0]:
async def get_highscores_world(session, limiter, world):
    first_page = await get_highscores(session, limiter, world, page=1)
    total_pages = first_page["highscores"]["highscore_page"]["total_pages"]

    tasks = [
        get_highscores(session, limiter, world, page=page)
        for page in range(2, total_pages + 1)
    ]

    results = await asyncio.gather(*tasks, return_exceptions=True)
    pages = [first_page]
    pages.extend(result for result in results if not isinstance(result, Exception))

    return pages

async def get_highscores_world(session, limiter, world):
    first_page = await get_highscores(session, limiter, world, page=1)
    total_pages = first_page["highscores"]["highscore_page"]["total_pages"]
    page_numbers = range(2, total_pages + 1)
 
    results = await asyncio.gather(
        *(get_highscores(session, limiter, world, page=p) for p in page_numbers),
        return_exceptions=True,
    )
    failed_pages = [p for p, r in zip(page_numbers, results) if isinstance(r, Exception)]
    if failed_pages:
        raise RuntimeError(f"{world}: pages {failed_pages} failed; world will not be saved in an incomplete state.")
 
    return [first_page, *results]


In [0]:
def get_world_names():
    worlds_df = spark.sql("""
      SELECT DISTINCT
             world.name AS world_name
        FROM tibia_analytics.bronze.raw_worlds
     LATERAL VIEW EXPLODE(worlds.regular_worlds) worlds_table AS world
       WHERE source_file_date = (
               SELECT MAX(source_file_date)
                 FROM tibia_analytics.bronze.raw_worlds
             )
    """)
    
    worlds = [row.world_name for row in worlds_df.collect()]
  
    if not worlds:
        raise RuntimeError(
            "No worlds found in bronze.raw_worlds for highscores ingestion."
        )

    return worlds

In [0]:
def get_characters_name():
    highscores_df = spark.sql("""
      SELECT DISTINCT
             highscore.world AS highscore_world,
             highscore.name AS highscore_name            
        FROM tibia_analytics.bronze.raw_highscores
     LATERAL VIEW EXPLODE(highscores.highscore_list) highscores_table AS highscore
       WHERE source_file_date = (
               SELECT MAX(source_file_date) 
                 FROM tibia_analytics.bronze.raw_highscores
             )
    """)
    highscores = [(row.highscore_world, row.highscore_name) for row in highscores_df.collect()]
  
    if not highscores:
        raise RuntimeError(
            "No characters found in bronze.raw_highscores for characters ingestion."
        )

    return highscores

In [0]:
def build_character_tasks(identity_rows, highscore_rows, world_list=None):
    worlds = None if world_list is None else set(world_list)
 
 
    def in_scope(world):
        return worlds is None or world in worlds
 
    locked_names, tasks = set(), {}
 
    for character_id, name, world, locked in identity_rows:
        locked_names.update(locked or ())
        if in_scope(world):
            tasks.setdefault(name.casefold(), (character_id, name, world))
 
    for world, name in highscore_rows:
        if in_scope(world) and name not in locked_names:
            tasks.setdefault(name.casefold(), (None, name, world))
 
    return list(tasks.values())

In [0]:
def _real_world(response, default):
    character = (response.get("character") or {}).get("character") or {}
    return character.get("world", default)
 
 
async def fetch_characters(session, limiter, items, num_workers=MAX_CONCURRENCY, report_every=5000):
    queue = asyncio.Queue()
    for item in items:
        queue.put_nowait(item)
    for _ in range(num_workers):
        queue.put_nowait(None)
 
    by_world, not_found, failed = {}, [], []
    counter, started, total = itertools.count(1), time.monotonic(), len(items)
 
    async def worker():
        while (item := await queue.get()) is not None:
            _, name, world = item
            try:
                response = await get_characters(session, limiter, name)
            except Exception as e:
                failed.append((item, f"{type(e).__name__}: {e}"))
            else:
                if response is None:
                    not_found.append(name)
                else:
                    by_world.setdefault(_real_world(response, world), []).append(response)
 
            done = next(counter)
            if done % report_every == 0:
                print(f"{done}/{total} | {done / (time.monotonic() - started):.1f} req/s "
                      f"| 404={len(not_found)} failed={len(failed)}")
 
    await asyncio.gather(*(worker() for _ in range(num_workers)))
    return by_world, not_found, failed

In [0]:
async def run_worlds_ingestion():
    timeout = aiohttp.ClientTimeout(total=15, connect=5)
    connector = aiohttp.TCPConnector(limit=32, limit_per_host=32, ttl_dns_cache=300)
    date_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")
   
    async with aiohttp.ClientSession(headers=headers, timeout=timeout, connector=connector) as session:
        data = await get_worlds(session, limiter)
        await save_async(data, "worlds", f"worlds_{date_str}.json")

In [0]:
async def run_highscores_ingestion(world_list):
    timeout = aiohttp.ClientTimeout(total=15, connect=5)
    connector = aiohttp.TCPConnector(limit=32, limit_per_host=32, ttl_dns_cache=300)
    date_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    async with aiohttp.ClientSession(headers=headers, timeout=timeout, connector=connector) as session:
        tasks = [
            get_highscores_world(session, limiter, world)
            for world in world_list
        ]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        save_tasks = []

        for world, data in zip(world_list, results):
            if isinstance(data, Exception):
                print(f"Error fetching {world}: {data}")
                continue

            if data:
                save_tasks.append(
                    save_async(data, f"highscores/{date_str}", f"{world}.json")
                )

        if save_tasks:
            await asyncio.gather(*save_tasks)

In [0]:
CHARACTERS_RETRY_COOLDOWN = 60
MAX_FAILURE_RATIO = 0.01
 
def get_identity_rows():
    identity_df = spark.sql("""
      SELECT character_id,
             current_name,
             current_world,
             locked_names 
        FROM tibia_analytics.silver.characters_identity
       WHERE is_deleted = FALSE
          OR deleted_at > CURRENT_TIMESTAMP()
    """)
    return [
        (r.character_id, r.current_name, r.current_world, r.locked_names)
        for r in identity_df.collect()
    ]
 

async def get_world_characters(world_list=None, limit=None, max_concurrency=MAX_CONCURRENCY, seed=42):
    global concurrency_limit
    concurrency_limit = asyncio.Semaphore(max_concurrency)
    http_stats.clear()
    latencies.clear()
 
    timeout = aiohttp.ClientTimeout(total=30, connect=5)
    connector = aiohttp.TCPConnector(limit=max_concurrency, limit_per_host=max_concurrency, ttl_dns_cache=300)
    date_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")
 
    tasks = build_character_tasks(get_identity_rows(), get_characters_name(), world_list)
    if limit is not None:
        tasks = random.Random(seed).sample(tasks, min(limit, len(tasks)))
    print(f"{len(tasks)} characters to colect (max_concurrency={max_concurrency})")
    report_every = max(1, len(tasks) // 10)
 
    started = time.monotonic()
    async with aiohttp.ClientSession(headers=headers, timeout=timeout, connector=connector) as session:
        by_world, not_found, failed = await fetch_characters(
            session, limiter, tasks, max_concurrency, report_every
        )
        print_http_summary(time.monotonic() - started)
 
        if failed:
            await asyncio.sleep(CHARACTERS_RETRY_COOLDOWN)
            retried, not_found_2, failed = await fetch_characters(
                session, limiter, [item for item, _ in failed], max_concurrency, report_every
            )
            for world, rows in retried.items():
                by_world.setdefault(world, []).extend(rows)
            not_found += not_found_2
 
    await asyncio.gather(
        *(save_async(rows, f"characters/{date_str}", f"{world}.json") for world, rows in by_world.items()),
        save_async({"not_found": not_found, "failed": failed}, f"characters_errors/{date_str}", "summary.json"),
    )
 
    if len(failed) > MAX_FAILURE_RATIO * len(tasks):
        raise RuntimeError(f"{len(failed)} out of {len(tasks)} characters failed after 2 passes.")

In [0]:
def run_characters_test(limit=2000, max_concurrency=MAX_CONCURRENCY, world_list=None,
                         seed=42, test_dir="/Volumes/tibia_analytics/bronze/scratch/tibia_test/"):
    global base_dir
    prod_base_dir, base_dir = base_dir, test_dir
    try:
        asyncio.run(get_world_characters(world_list, limit=limit, max_concurrency=max_concurrency, seed=seed))
    finally:
        base_dir = prod_base_dir